# 01 — Premiers pas avec les qubits

Objectif de ce notebook : comprendre **juste assez** de physique quantique pour construire un classifieur quantique.

1. Un qubit, la superposition et la mesure
2. Les rotations : régler une probabilité avec un angle
3. L'intrication : deux qubits corrélés
4. Encoder une donnée dans un circuit
5. Notre premier classifieur quantique

> Noyau à sélectionner : **Python (quantum-classifier)** (créé par `setup.bat`).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))  # pour importer le paquet qvc du projet

import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml

print("PennyLane", qml.__version__)

## 1. Un qubit

Un bit classique vaut 0 **ou** 1. Un qubit est dans un état
$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1.$$

Quand on le **mesure**, on obtient 0 avec la probabilité $|\alpha|^2$ et 1 avec la probabilité $|\beta|^2$.
La porte de **Hadamard** $H$ transforme $|0\rangle$ en $\frac{1}{\sqrt 2}(|0\rangle + |1\rangle)$ : une superposition 50/50.

In [ ]:
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def superposition():
    qml.Hadamard(wires=0)
    return qml.probs(wires=0)

print("Probabilités exactes [P(0), P(1)] :", superposition())
print(qml.draw(superposition)())

Sur un vrai ordinateur quantique, on n'a pas accès aux probabilités : on répète l'expérience (des *shots*) et on compte.
Plus il y a de shots, plus l'estimation est précise — c'est le **bruit statistique** de la mesure.

In [ ]:
for shots in (10, 100, 10_000):
    dev_shots = qml.device("default.qubit", wires=1)

    @qml.set_shots(shots)
    @qml.qnode(dev_shots)
    def mesure():
        qml.Hadamard(wires=0)
        return qml.counts(wires=0)

    print(f"{shots:>6} shots ->", {k: int(v) for k, v in mesure().items()})

## 2. Les rotations

La porte $R_Y(\theta)$ fait tourner l'état d'un angle $\theta$. En partant de $|0\rangle$ :
$$P(1) = \sin^2(\theta/2).$$
C'est le **bouton de réglage** des circuits quantiques : un angle continu, donc dérivable, donc entraînable.

In [ ]:
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def rotation(theta):
    qml.RY(theta, wires=0)
    return qml.probs(wires=0)

thetas = np.linspace(0, 2 * np.pi, 100)
p1 = [rotation(t)[1] for t in thetas]

plt.figure(figsize=(6, 3))
plt.plot(thetas, p1, label="circuit")
plt.plot(thetas, np.sin(thetas / 2) ** 2, "--", label=r"$\sin^2(\theta/2)$")
plt.xlabel(r"$\theta$"); plt.ylabel("P(1)"); plt.legend(); plt.tight_layout(); plt.show()

## 3. L'intrication

Avec deux qubits, $H$ puis une porte **CNOT** produisent l'état de Bell
$$|\Phi^+\rangle = \tfrac{1}{\sqrt 2}(|00\rangle + |11\rangle).$$
Chaque qubit, pris seul, est aléatoire… mais les deux donnent **toujours le même résultat**.
Aucun état classique de deux bits indépendants ne reproduit cela.

In [ ]:
dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def bell():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])

for etat, p in zip(["00", "01", "10", "11"], bell()):
    print(f"P({etat}) = {p:.2f}")
print(qml.draw(bell)())

### Le même circuit avec Qiskit

Le projet utilise PennyLane pour l'apprentissage, mais Qiskit est la bibliothèque d'IBM (et celle de leurs vrais processeurs).
Voici le même état de Bell, simulé avec Qiskit Aer.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)
qc.measure([0, 1], [0, 1])
print(qc.draw())

resultat = AerSimulator().run(qc, shots=1000).result()
print("Comptages :", resultat.get_counts())

## 4. Encoder une donnée

Pour qu'un circuit « voie » une donnée $x = (x_1, \dots, x_n)$, on l'encode en angles :
le qubit $i$ subit $R_Y(x_i)$. C'est l'**encodage en angles** (`AngleEmbedding`).
C'est pour cela que `load_dataset` remet les variables dans $[0, \pi]$, et qu'il faut **un qubit par variable**.

In [ ]:
dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def encode(x):
    qml.AngleEmbedding(x, wires=[0, 1], rotation="Y")
    return qml.probs(wires=[0, 1])

x = np.array([np.pi / 2, np.pi / 4])
print(qml.draw(encode)(x))
print("Probabilités :", np.round(encode(x), 3))

## 5. Notre premier classifieur quantique

Le **classifieur quantique variationnel** (VQC) enchaîne :
encodage de $x$ → couches de rotations **à paramètres entraînables** + CNOT → mesure de $\langle Z \rangle \in [-1, 1]$.
On prédit la classe selon le signe, et on ajuste les angles par descente de gradient — exactement comme un réseau de neurones.

In [ ]:
from qvc import VariationalQuantumClassifier, load_dataset

X_train, X_test, y_train, y_test = load_dataset("moons", n_features=2)
vqc = VariationalQuantumClassifier(n_layers=2, steps=60, seed=0, verbose=True)
vqc.fit(X_train, y_train)

print(f"\nAccuracy test : {vqc.score(X_test, y_test):.3f}  |  paramètres : {vqc.n_params_}")
print(vqc.draw())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(vqc.loss_history_)
axes[0].set_title("Perte pendant l'entraînement"); axes[0].set_xlabel("étape")

g = np.linspace(0, np.pi, 30)
xx, yy = np.meshgrid(g, g)
zz = vqc.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[1].contourf(xx, yy, zz, levels=20, cmap="RdBu", alpha=0.7)
axes[1].scatter(*X_test.T, c=y_test, cmap="RdBu", edgecolors="k")
axes[1].set_title("Frontière de décision (données de test)")
plt.tight_layout(); plt.show()

## Et face aux modèles classiques ?

Lancez `python run_benchmark.py` dans un terminal (ou la cellule suivante) : vous verrez que sur ces petits problèmes,
les modèles classiques font **au moins aussi bien, beaucoup plus vite**. C'est normal — et c'est tout l'objet du projet :
mesurer honnêtement où en est l'apprentissage quantique.

## Exercices

1. Passez `n_layers` de 1 à 4 : l'accuracy progresse-t-elle ? Et le temps d'entraînement ?
2. Remplacez `"moons"` par `"circles"`. Pourquoi est-ce plus difficile pour le VQC ?
3. Dans la partie 1, combien de shots faut-il pour estimer P(1) à ±1 % près ?
4. Ajoutez du bruit : créez `VariationalQuantumClassifier(shots=100)`. Que devient la courbe de perte ?

In [ ]:
from qvc import run_benchmark

df = run_benchmark("moons", n_qubits=2, out_dir=Path.cwd().parent / "results", verbose=False)
df